# LeetCode 444 — Sequence Reconstruction bằng Python

Link đề: https://leetcode.com/problems/sequence-reconstruction/ · Độ khó: **Medium** · Chủ đề: Graph, Topological Sort, Array, Directed Acyclic Graph

Skeleton LeetCode:

```python
class Solution:
    def sequenceReconstruction(self, nums: List[int], sequences: List[List[int]]) -> bool:
        ...
```

Ghi chú: đây là bài **Premium** (cần tài khoản LeetCode Premium để xem đề gốc). Đề trong notebook được viết lại theo hiểu biết về bài, nên **đối chiếu lại đề gốc** nếu bạn có tài khoản Premium. Các ví dụ và ràng buộc dưới đây là phiên bản mà mọi lời giải trong notebook đã được kiểm chứng bằng code (đối chiếu với vét cạn mọi hoán vị).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho hoán vị `nums` của `1..n` và một số dãy con `sequences` (mỗi dãy là dãy con của `nums`); hỏi `nums` có phải là **siêu dãy ngắn nhất và duy nhất** của tất cả các dãy đó hay không.
2. Nắm các công cụ Python: kiểm tra dãy con, `zip(seq, seq[1:])` để lấy các cặp kề nhau, danh sách kề `g[a].append(b)`, mảng bậc vào `indeg`, `collections.deque`, `itertools.permutations`, và `set` các cặp `(a, b)`.
3. Đi từ **vét cạn mọi hoán vị `O(n! · ...)`** → **sắp xếp topo (Kahn) với điều kiện "hàng đợi luôn đúng một phần tử"** → **chỉ cần kiểm tra các cặp kề nhau của `nums` có "cạnh trực tiếp" hay không** → **Kahn so từng bước với `nums`**.
4. Hiểu mấu chốt: mỗi dãy con cho các ràng buộc "`a` đứng trước `b`"; các ràng buộc tạo thành đồ thị có hướng không chu trình (DAG); `nums` là **duy nhất** khi và chỉ khi tại mọi bước **chỉ có đúng một đỉnh bậc vào bằng 0** — tương đương mọi cặp kề nhau `nums[i], nums[i+1]` đều được "ép" trực tiếp bởi một dãy.
5. Tránh các lỗi kinh điển: chỉ kiểm tra "sắp xếp topo được" mà quên tính **duy nhất**, bỏ sót phần tử **không xuất hiện** trong dãy nào, nhầm chỉ số khi giá trị bắt đầu từ `1`, và đếm số cạnh thay vì kiểm tra **đúng các cặp kề trong `nums`**.

## 2. Đề bài & ví dụ

Cho một mảng số nguyên `nums` có độ dài `n`, là một **hoán vị** của các số `1..n`. Cho thêm một mảng hai chiều `sequences`, trong đó mỗi `sequences[i]` là một **dãy con** của `nums`.

Hãy kiểm tra xem `nums` có phải là **siêu dãy ngắn nhất và duy nhất** của `sequences` hay không.

- **Siêu dãy** (supersequence) của `sequences` là một dãy mà **mọi** `sequences[i]` đều là dãy con của nó.
- **Siêu dãy ngắn nhất** là siêu dãy có độ dài nhỏ nhất. Có thể có **nhiều** siêu dãy ngắn nhất.
- **Dãy con** của một dãy là dãy thu được bằng cách xóa một số (hoặc không xóa) phần tử mà **giữ nguyên thứ tự** các phần tử còn lại.

Trả về `true` nếu `nums` là siêu dãy ngắn nhất **duy nhất** của `sequences`, ngược lại trả về `false`.

**Ví dụ 1**

- Input: `nums = [1,2,3]`, `sequences = [[1,2],[1,3]]`
- Output: `false`
- Giải thích: có hai siêu dãy ngắn nhất: `[1,2,3]` và `[1,3,2]` (cả hai đều chứa `[1,2]` và `[1,3]` làm dãy con). `nums` không duy nhất.

**Ví dụ 2**

- Input: `nums = [1,2,3]`, `sequences = [[1,2]]`
- Output: `false`
- Giải thích: siêu dãy ngắn nhất chỉ là `[1,2]` (dài `2`), ngắn hơn `nums` (dài `3`). `nums` không phải ngắn nhất.

**Ví dụ 3**

- Input: `nums = [1,2,3]`, `sequences = [[1,2],[1,3],[2,3]]`
- Output: `true`
- Giải thích: siêu dãy ngắn nhất duy nhất là `[1,2,3]`.

**Ràng buộc:**

- `n == nums.length`, `1 <= n <= 10^4`
- `nums` là một hoán vị của `1..n`
- `1 <= sequences.length <= 10^4`, `1 <= sequences[i].length <= 10^4`
- `1 <= sum(sequences[i].length) <= 10^5`
- `1 <= sequences[i][j] <= n`
- Các dãy trong `sequences` đôi một **khác nhau**
- Mỗi `sequences[i]` là dãy con của `nums`

In [1]:
from typing import List
from collections import deque
from itertools import permutations
import random
import time

examples = [
    ([1, 2, 3], [[1, 2], [1, 3]], False),
    ([1, 2, 3], [[1, 2]], False),
    ([1, 2, 3], [[1, 2], [1, 3], [2, 3]], True),
]
for nums, seqs, expected in examples:
    print("nums =", nums, "sequences =", seqs, "-> kỳ vọng", expected)

nums = [1, 2, 3] sequences = [[1, 2], [1, 3]] -> kỳ vọng False
nums = [1, 2, 3] sequences = [[1, 2]] -> kỳ vọng False
nums = [1, 2, 3] sequences = [[1, 2], [1, 3], [2, 3]] -> kỳ vọng True


## 3. Kiến thức nền

### 3.1. Dãy con (subsequence) và hoán vị (permutation)

`[1, 3]` là dãy con của `[1, 2, 3]` (xóa `2`, giữ nguyên thứ tự `1` trước `3`), còn `[3, 1]` thì **không** (sai thứ tự). **Hoán vị** của `1..n` là cách sắp xếp lại `n` số đó (mỗi số đúng một lần). Cách kiểm tra "dãy con" gọn: duyệt `nums` bằng một iterator, và mỗi phần tử của dãy phải được "tìm thấy" tiếp theo.

In [2]:
def is_subsequence(seq, arr):
    it = iter(arr)
    return all(x in it for x in seq)      # `x in it` tiêu thụ iterator đến khi gặp x


print(is_subsequence([1, 3], [1, 2, 3]))      # True
print(is_subsequence([3, 1], [1, 2, 3]))      # False
print(is_subsequence([], [1, 2, 3]))          # True (dãy rỗng)

True
False
True


### 3.2. Siêu dãy ngắn nhất và ràng buộc thứ tự

Mỗi dãy con, ví dụ `[1, 2, 5]`, nói rằng "`1` đứng trước `2`, `2` đứng trước `5`". Một siêu dãy phải **tôn trọng mọi ràng buộc đó**. Muốn dãy ngắn nhất thì **không thêm phần tử thừa**: chỉ gồm các số xuất hiện trong ít nhất một `sequences[i]`. Nếu có số `1..n` nào chưa từng xuất hiện trong `sequences`, siêu dãy ngắn nhất sẽ **không chứa** số đó và vì thế ngắn hơn `nums`.

In [3]:
seqs = [[1, 2], [1, 3]]
appear = set(x for s in seqs for x in s)
print("các số có mặt:", appear)             # {1, 2, 3}: siêu dãy ngắn nhất dài 3
seqs2 = [[1, 2]]
print("các số có mặt:", set(x for s in seqs2 for x in s), "-> siêu dãy ngắn nhất chỉ dài 2")

các số có mặt: {1, 2, 3}
các số có mặt: {1, 2} -> siêu dãy ngắn nhất chỉ dài 2


### 3.3. Cặp kề nhau: `zip(seq, seq[1:])`

Từ một dãy `[a0, a1, a2, ...]` ta chỉ cần các cặp **kề nhau** `(a0, a1), (a1, a2), ...` để biết toàn bộ thứ tự (vì tính bắc cầu: `a0 < a1 < a2` suy ra `a0 < a2`). `zip(seq, seq[1:])` ghép từng phần tử với phần tử ngay sau nó.

In [4]:
seq = [1, 2, 5, 7]
print(list(zip(seq, seq[1:])))      # [(1, 2), (2, 5), (5, 7)]
print(list(zip([4], [4][1:])))      # [] : dãy một phần tử không có cặp nào

[(1, 2), (2, 5), (5, 7)]
[]


### 3.4. Đồ thị có hướng, danh sách kề và bậc vào

Mỗi cặp `(a, b)` là một **cạnh có hướng** `a → b` ("a đứng trước b"). Ta lưu:

- `g[a]` = danh sách các đỉnh `b` sao cho có cạnh `a → b` (danh sách kề);
- `indeg[b]` = số cạnh đi **vào** `b` (bậc vào).

Vì giá trị chạy `1..n`, ta cấp mảng dài `n + 1` và bỏ ô `0` để dùng giá trị làm chỉ số trực tiếp (tránh nhầm lệch `1`).

In [5]:
n = 3
g = [[] for _ in range(n + 1)]       # g[0] không dùng
indeg = [0] * (n + 1)
for seq in [[1, 2], [1, 3], [2, 3]]:
    for a, b in zip(seq, seq[1:]):
        g[a].append(b)
        indeg[b] += 1
print("g     =", g)
print("indeg =", indeg)

g     = [[], [2, 3], [3], []]
indeg = [0, 0, 1, 2]


### 3.5. Sắp xếp topo (Kahn) và `deque`

**Sắp xếp topo** là cách xếp các đỉnh của một đồ thị có hướng không chu trình (DAG) thành một hàng sao cho mọi cạnh `a → b` có `a` đứng trước `b`. **Thuật toán Kahn**:

1. Đưa mọi đỉnh có `indeg == 0` vào hàng đợi.
2. Lấy một đỉnh `u` ra, "xóa" nó: với mỗi `v` trong `g[u]`, giảm `indeg[v]`; nếu về `0` thì đưa `v` vào hàng đợi.
3. Lặp lại đến khi hàng đợi rỗng.

`collections.deque` là hàng đợi hai đầu: `append` thêm cuối, `popleft` lấy đầu, đều `O(1)` (nhanh hơn `list.pop(0)`).

In [6]:
q = deque([1])
q.append(5)
q.append(7)
print(q, len(q))
print(q.popleft(), q)         # 1 deque([5, 7])

deque([1, 5, 7]) 3
1 deque([5, 7])


### 3.6. Tập các cặp và `itertools.permutations`

Một `set` có thể chứa các **tuple** như `(1, 2)`; kiểm tra `(1, 2) in pairs` là `O(1)` trung bình. `permutations(iterable)` sinh mọi hoán vị (có `n!` cái) — chỉ dùng được với `n` rất nhỏ, đúng cho vét cạn kiểm chứng.

In [7]:
pairs = {(1, 2), (2, 3)}
print((1, 2) in pairs, (1, 3) in pairs)
print(list(permutations([1, 2, 3]))[:3], "...", len(list(permutations([1, 2, 3, 4]))))   # 4! = 24

True False
[(1, 2, 3), (1, 3, 2), (2, 1, 3)] ... 24


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Xét ví dụ 1: `nums = [1,2,3]`, `sequences = [[1,2],[1,3]]`. Hai dãy nói: "`1` trước `2`" và "`1` trước `3`". Còn **chưa** có thông tin nào về thứ tự của `2` và `3`. Vậy cả `[1,2,3]` và `[1,3,2]` đều hợp lệ ⇒ không duy nhất ⇒ `false`.

Xét ví dụ 3: thêm `[2,3]` ⇒ "`2` trước `3`". Giờ ba ràng buộc `1<2`, `1<3`, `2<3` ép thứ tự duy nhất `1,2,3` ⇒ `true`.

Cách nghĩ: **"Ai được phép đứng đầu?"**. Ở mỗi bước, ứng viên đứng đầu là những số **không bị số chưa xếp nào đứng trước** (bậc vào bằng `0`). Nếu **luôn chỉ có đúng một ứng viên**, thứ tự bị ép hoàn toàn (duy nhất). Nếu có lúc có hai ứng viên, ta có thể chọn cái nào trước cũng được ⇒ có ít nhất hai thứ tự ⇒ không duy nhất.

| Bước | Số chưa xếp | Ứng viên đứng đầu (bậc vào 0) | Kết luận |
|---|---|---|---|
| 1 | 1, 2, 3 | 1 | duy nhất, xếp 1 |
| 2 | 2, 3 | 2 và 3 (ví dụ 1) | **hai ứng viên ⇒ false** |
| 2' | 2, 3 | chỉ 2 (ví dụ 3, vì `2 < 3`) | duy nhất, xếp 2 |
| 3' | 3 | 3 | duy nhất, xếp 3 ⇒ **true** |

In [8]:
def candidates(n, sequences, placed):
    # các số chưa xếp mà không bị số nào chưa xếp đứng trước
    before = {v: set() for v in range(1, n + 1)}
    for seq in sequences:
        for a, b in zip(seq, seq[1:]):
            before[b].add(a)
    return [v for v in range(1, n + 1) if v not in placed and not (before[v] - placed)]


for nums, seqs, expected in examples:
    placed = set()
    print("nums =", nums, "seqs =", seqs)
    for step in range(len(nums)):
        c = candidates(len(nums), seqs, placed)
        print("  bước", step + 1, "ứng viên:", c)
        if len(c) != 1:
            break
        placed.add(c[0])

nums = [1, 2, 3] seqs = [[1, 2], [1, 3]]
  bước 1 ứng viên: [1]
  bước 2 ứng viên: [2, 3]
nums = [1, 2, 3] seqs = [[1, 2]]
  bước 1 ứng viên: [1, 3]
nums = [1, 2, 3] seqs = [[1, 2], [1, 3], [2, 3]]
  bước 1 ứng viên: [1]
  bước 2 ứng viên: [2]
  bước 3 ứng viên: [3]


## 5. Cách 1 — Brute Force / baseline: thử mọi hoán vị

**Ý tưởng.** Siêu dãy ngắn nhất chứa đúng các số xuất hiện trong `sequences`. Ta liệt kê **mọi hoán vị** của các số đó, giữ lại những hoán vị mà mọi dãy trong `sequences` là dãy con của nó, rồi xem có đúng **một** hoán vị như vậy và hoán vị đó có bằng `nums` hay không.

Kiểm tra "dãy `s` là dãy con của hoán vị `perm`" nhanh bằng bảng vị trí `pos[v]`: dãy con khi và chỉ khi với mọi cặp kề `(a, b)` ta có `pos[a] < pos[b]` (hoán vị không có phần tử lặp nên điều đó đủ).

In [9]:
def seq_brute(nums, sequences):
    elems = sorted(set(x for s in sequences for x in s))
    valid = []
    for perm in permutations(elems):
        pos = {v: i for i, v in enumerate(perm)}
        if all(pos[a] < pos[b] for s in sequences for a, b in zip(s, s[1:])):
            valid.append(list(perm))
            if len(valid) > 1:
                return False                 # đã có hai siêu dãy ngắn nhất
    return len(valid) == 1 and valid[0] == nums


for nums, seqs, expected in examples:
    print(seq_brute(nums, seqs), "(kỳ vọng", expected, ")")

False (kỳ vọng False )
False (kỳ vọng False )
True (kỳ vọng True )


**Độ phức tạp (Cách 1).** Có tới `m!` hoán vị (`m` = số phần tử khác nhau, tối đa `n`), mỗi hoán vị kiểm tra `O(tổng độ dài)`: **Time `O(n! · L)`** với `L = sum(len(seq))`. **Space `O(n)`** (một hoán vị + bảng `pos`). Chỉ dùng được với `n <= 8` hay hơn một chút.

**Nút thắt:** ta đang đếm hoán vị, trong khi chỉ cần biết "**có lúc nào có hai lựa chọn không**". Thuật toán Kahn giúp trả lời điều đó mà không liệt kê.

## 6. Ý tưởng thuật toán / bước đột phá

Biến bài toán thành **sắp xếp topo**:

- Mỗi cặp kề `(a, b)` trong một dãy là cạnh `a → b`.
- Một siêu dãy chứa mọi dãy là một **thứ tự topo** của đồ thị (thứ tự các đỉnh tôn trọng mọi cạnh). `nums` luôn là một thứ tự topo vì mỗi dãy là dãy con của `nums`.
- Có **duy nhất** một thứ tự topo ⇔ ở mọi thời điểm của thuật toán Kahn, hàng đợi chứa **đúng một** đỉnh. Vì nếu có hai đỉnh `x`, `y` cùng trong hàng đợi, ta có thể đưa `x` ra trước hoặc `y` ra trước, cho hai thứ tự khác nhau.
- Nếu duy nhất thì thứ tự duy nhất đó **chính là `nums`** (vì `nums` hợp lệ) — không cần so sánh từng vị trí với `nums`.
- Số nào **không xuất hiện** trong dãy nào sẽ có bậc vào `0` ngay từ đầu và **không bao giờ bị ép**, nên nó làm hàng đợi có `>= 2` đỉnh (hoặc `n = 1` nhưng khi đó dãy nào cũng chứa `1`) — Kahn tự động "bắt" luôn lỗi này.

Từ đây có ba lời giải: Kahn với điều kiện hàng đợi đúng một phần tử (Cách 2), kiểm tra trực tiếp các **cặp kề của `nums`** (Cách 3), và Kahn so từng đỉnh với `nums` (Cách 4).

## 7. Các cách tiếp theo

Tổng cộng **4 cách**: (1) vét cạn hoán vị ở mục 5; (2) Kahn, hàng đợi luôn đúng một đỉnh; (3) kiểm tra các cặp kề của `nums`; (4) Kahn so từng đỉnh với `nums`. Mỗi cách: giải thích → code demo → code nộp LeetCode → độ phức tạp.

### 7.1. Cách 2 — Kahn: hàng đợi luôn có đúng một đỉnh

**Giải thích từng đoạn.**

- Dựng `g` và `indeg` từ các cặp kề của mọi dãy (mảng dài `n + 1`, dùng giá trị làm chỉ số).
- `q` chứa mọi đỉnh `v` trong `1..n` có `indeg[v] == 0`.
- `while len(q) == 1:` — chừng nào còn **đúng một** ứng viên: lấy `u` ra, với mỗi `v` trong `g[u]` giảm `indeg[v]`, nếu `0` thì đưa vào `q`.
- Khi vòng lặp dừng, `len(q)` là `0` (đã xếp hết, thứ tự duy nhất ⇒ `True`) hoặc `>= 2` (có lúc hai ứng viên ⇒ `False`).

Vì sao `len(q)` không thể là `0` giữa chừng khi còn đỉnh chưa xếp? Vì đồ thị là DAG (do mọi dãy là dãy con của cùng một hoán vị `nums`, không thể có chu trình) nên luôn có đỉnh bậc vào `0` cho đến khi xếp hết.

In [10]:
def seq_kahn(nums, sequences):
    n = len(nums)
    g = [[] for _ in range(n + 1)]
    indeg = [0] * (n + 1)
    for seq in sequences:
        for a, b in zip(seq, seq[1:]):
            g[a].append(b)
            indeg[b] += 1
    q = deque(v for v in range(1, n + 1) if indeg[v] == 0)
    while len(q) == 1:
        u = q.popleft()
        for v in g[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return len(q) == 0


for nums, seqs, expected in examples:
    print(seq_kahn(nums, seqs), "(kỳ vọng", expected, ")")

False (kỳ vọng False )
False (kỳ vọng False )
True (kỳ vọng True )


In [11]:
class SolutionKahn:
    def sequenceReconstruction(self, nums: List[int], sequences: List[List[int]]) -> bool:
        n = len(nums)
        g = [[] for _ in range(n + 1)]
        indeg = [0] * (n + 1)
        for seq in sequences:
            for a, b in zip(seq, seq[1:]):
                g[a].append(b)
                indeg[b] += 1
        q = deque(v for v in range(1, n + 1) if indeg[v] == 0)
        while len(q) == 1:
            u = q.popleft()
            for v in g[u]:
                indeg[v] -= 1
                if indeg[v] == 0:
                    q.append(v)
        return len(q) == 0


print(SolutionKahn().sequenceReconstruction([1, 2, 3], [[1, 2], [1, 3], [2, 3]]))

True


**Độ phức tạp (Cách 2).** Dựng đồ thị `O(L)` với `L = sum(len(seq))`; mỗi đỉnh vào/ra hàng đợi tối đa một lần, mỗi cạnh xử lý một lần: **Time `O(n + L)`**, **Space `O(n + L)`** cho danh sách kề (tối đa `L` cạnh).

### 7.2. Cách 3 — Chỉ cần kiểm tra các cặp kề của `nums`

**Giải thích từng đoạn.**

Vì mọi dãy là dãy con của `nums`, mỗi cặp kề `(a, b)` trong một dãy luôn thỏa `a` đứng trước `b` trong `nums`. Thứ tự topo duy nhất ⇔ với **mọi cặp kề** `(nums[i], nums[i+1])` có **một dãy nào đó chứa nó như một cạnh trực tiếp** (cặp kề của dãy đó).

Vì sao?

- (⇐) Nếu mọi `nums[i] → nums[i+1]` là cạnh, thì trong bất kỳ thứ tự topo nào, `nums[0]` phải trước `nums[1]`, `nums[1]` trước `nums[2]`, ... ⇒ thứ tự bắt buộc là `nums`. Duy nhất.
- (⇒) Nếu thiếu cạnh `nums[i] → nums[i+1]`, ta hoán đổi hai phần tử kề đó trong `nums` thu được thứ tự khác; nó vẫn hợp lệ vì không có ràng buộc trực tiếp giữa hai phần tử (các ràng buộc gián tiếp đi qua phần tử thứ ba nằm xen giữa trong `nums` — mà ở đây hai phần tử kề nhau nên không có phần tử xen giữa). Vậy có thứ tự thứ hai ⇒ không duy nhất.

Ngoài ra phải xét phần tử **không xuất hiện** trong dãy nào (làm siêu dãy ngắn hơn): với `n >= 2`, điều kiện mọi cặp kề đều có cạnh đã tự động phủ mọi phần tử; với `n == 1` ta cần dãy chứa `1` (đề đảm bảo). Để an toàn ta kiểm tra thêm tập phần tử có mặt bằng `set(nums)`.

In [12]:
def seq_adjacent(nums, sequences):
    pairs = set()
    seen = set()
    for seq in sequences:
        seen.update(seq)
        for a, b in zip(seq, seq[1:]):
            pairs.add((a, b))
    if seen != set(nums):                       # có số không xuất hiện -> siêu dãy ngắn nhất ngắn hơn nums
        return False
    return all((nums[i], nums[i + 1]) in pairs for i in range(len(nums) - 1))


for nums, seqs, expected in examples:
    print(seq_adjacent(nums, seqs), "(kỳ vọng", expected, ")")

False (kỳ vọng False )
False (kỳ vọng False )
True (kỳ vọng True )


In [13]:
class SolutionAdjacent:
    def sequenceReconstruction(self, nums: List[int], sequences: List[List[int]]) -> bool:
        pairs = set()
        seen = set()
        for seq in sequences:
            seen.update(seq)
            for a, b in zip(seq, seq[1:]):
                pairs.add((a, b))
        if seen != set(nums):
            return False
        return all((nums[i], nums[i + 1]) in pairs for i in range(len(nums) - 1))


print(SolutionAdjacent().sequenceReconstruction([1, 2, 3], [[1, 2], [1, 3]]))

False


**Độ phức tạp (Cách 3).** Duyệt mọi cặp kề một lần `O(L)`, kiểm tra `n - 1` cặp `O(n)`: **Time `O(n + L)`**, **Space `O(n + L)`** cho tập cặp (tối đa `L` cặp). Không cần hàng đợi hay bậc vào — ngắn nhất, nhưng dựa trên **giả thiết đề bảo đảm** mọi dãy là dãy con của `nums`.

### 7.3. Cách 4 — Kahn so từng đỉnh với `nums`

**Giải thích từng đoạn.** Biến thể "sách giáo khoa": cũng dựng `g`, `indeg`, `q` như Cách 2, nhưng thay vì kiểm tra cuối cùng, ta **duyệt `nums` theo thứ tự**: tại bước `i`, hàng đợi phải có **đúng một** phần tử và nó phải **bằng `nums[i]`**; nếu không, trả `False`. Hết `nums` mà không vi phạm ⇒ `True`.

Cách này không cần dựa vào việc "thứ tự duy nhất nhất thiết là `nums`" — nó **so trực tiếp** — nên an toàn hơn nếu bạn lo ngại giả thiết của đề.

In [14]:
def seq_kahn_compare(nums, sequences):
    n = len(nums)
    g = [[] for _ in range(n + 1)]
    indeg = [0] * (n + 1)
    for seq in sequences:
        for a, b in zip(seq, seq[1:]):
            g[a].append(b)
            indeg[b] += 1
    q = deque(v for v in range(1, n + 1) if indeg[v] == 0)
    for x in nums:
        if len(q) != 1 or q[0] != x:
            return False
        u = q.popleft()
        for v in g[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return True


for nums, seqs, expected in examples:
    print(seq_kahn_compare(nums, seqs), "(kỳ vọng", expected, ")")

False (kỳ vọng False )
False (kỳ vọng False )
True (kỳ vọng True )


In [15]:
class SolutionKahnCompare:
    def sequenceReconstruction(self, nums: List[int], sequences: List[List[int]]) -> bool:
        n = len(nums)
        g = [[] for _ in range(n + 1)]
        indeg = [0] * (n + 1)
        for seq in sequences:
            for a, b in zip(seq, seq[1:]):
                g[a].append(b)
                indeg[b] += 1
        q = deque(v for v in range(1, n + 1) if indeg[v] == 0)
        for x in nums:
            if len(q) != 1 or q[0] != x:
                return False
            u = q.popleft()
            for v in g[u]:
                indeg[v] -= 1
                if indeg[v] == 0:
                    q.append(v)
        return True


print(SolutionKahnCompare().sequenceReconstruction([1, 2, 3], [[1, 2]]))

False


**Độ phức tạp (Cách 4).** Giống Cách 2: **Time `O(n + L)`**, **Space `O(n + L)`**.

## 8. TRACE từng bước

### 8.1. Bảng trạng thái — Kahn cho ví dụ 3 (`nums = [1,2,3]`, `sequences = [[1,2],[1,3],[2,3]]`)

Đồ thị: `1 → 2`, `1 → 3`, `2 → 3`. Bậc vào: `indeg[1] = 0`, `indeg[2] = 1`, `indeg[3] = 2`.

| Bước | Hàng đợi `q` trước | `len(q)` | Lấy `u` | Giảm bậc vào | `q` sau |
|---|---|---|---|---|---|
| 0 | `[1]` | 1 | 1 | `indeg[2]: 1→0` (đưa 2 vào), `indeg[3]: 2→1` | `[2]` |
| 1 | `[2]` | 1 | 2 | `indeg[3]: 1→0` (đưa 3 vào) | `[3]` |
| 2 | `[3]` | 1 | 3 | (không có cạnh ra) | `[]` |
| kết thúc | `[]` | 0 | | | `len(q) == 0` ⇒ `True` |

Với ví dụ 1 (`[[1,2],[1,3]]`): `indeg = [_, 0, 1, 1]`; bước 0: `q = [1]` → lấy `1`, `indeg[2]: 1→0`, `indeg[3]: 1→0`, `q = [2, 3]`, `len(q) = 2` ⇒ dừng vòng lặp ⇒ `False`.

In [16]:
def trace_kahn(nums, sequences):
    n = len(nums)
    g = [[] for _ in range(n + 1)]
    indeg = [0] * (n + 1)
    for seq in sequences:
        for a, b in zip(seq, seq[1:]):
            g[a].append(b)
            indeg[b] += 1
    print("g     =", g[1:], "(g[1], g[2], ...)")
    print("indeg =", indeg[1:], "(indeg[1], indeg[2], ...)")
    q = deque(v for v in range(1, n + 1) if indeg[v] == 0)
    step = 0
    while len(q) == 1:
        print(f"bước {step}: q={list(q)} (đúng một ứng viên)")
        u = q.popleft()
        for v in g[u]:
            indeg[v] -= 1
            note = " -> đưa vào q" if indeg[v] == 0 else ""
            print(f"    xóa {u}: indeg[{v}] -> {indeg[v]}{note}")
            if indeg[v] == 0:
                q.append(v)
        step += 1
    print(f"dừng: q={list(q)} (len={len(q)})")
    return len(q) == 0


print(trace_kahn([1, 2, 3], [[1, 2], [1, 3], [2, 3]]))
print()
print(trace_kahn([1, 2, 3], [[1, 2], [1, 3]]))
print()
print(trace_kahn([1, 2, 3], [[1, 2]]))

g     = [[2, 3], [3], []] (g[1], g[2], ...)
indeg = [0, 1, 2] (indeg[1], indeg[2], ...)
bước 0: q=[1] (đúng một ứng viên)
    xóa 1: indeg[2] -> 0 -> đưa vào q
    xóa 1: indeg[3] -> 1
bước 1: q=[2] (đúng một ứng viên)
    xóa 2: indeg[3] -> 0 -> đưa vào q
bước 2: q=[3] (đúng một ứng viên)
dừng: q=[] (len=0)
True

g     = [[2, 3], [], []] (g[1], g[2], ...)
indeg = [0, 1, 1] (indeg[1], indeg[2], ...)
bước 0: q=[1] (đúng một ứng viên)
    xóa 1: indeg[2] -> 0 -> đưa vào q
    xóa 1: indeg[3] -> 0 -> đưa vào q
dừng: q=[2, 3] (len=2)
False

g     = [[2], [], []] (g[1], g[2], ...)
indeg = [0, 1, 0] (indeg[1], indeg[2], ...)
dừng: q=[1, 3] (len=2)
False


### 8.2. Trace Cách 3 (các cặp kề của `nums`)

Với ví dụ 3: `pairs = {(1,2), (1,3), (2,3)}`. Cặp kề của `nums = [1,2,3]` là `(1,2)` và `(2,3)`; cả hai đều trong `pairs` ⇒ `True`. Với ví dụ 1: `pairs = {(1,2), (1,3)}`; cặp `(2,3)` **không có** ⇒ `False` (đúng: thứ tự giữa `2` và `3` chưa được ép).

In [17]:
def trace_adjacent(nums, sequences):
    pairs = set()
    for seq in sequences:
        for a, b in zip(seq, seq[1:]):
            pairs.add((a, b))
    print("pairs =", sorted(pairs))
    ok = True
    for i in range(len(nums) - 1):
        pair = (nums[i], nums[i + 1])
        has = pair in pairs
        print(f"  cặp kề của nums: {pair} -> {'có' if has else 'THIẾU'}")
        ok = ok and has
    return ok


print(trace_adjacent([1, 2, 3], [[1, 2], [1, 3], [2, 3]]))
print(trace_adjacent([1, 2, 3], [[1, 2], [1, 3]]))
print(trace_adjacent([3, 1, 2, 4], [[3, 1], [1, 4], [2, 4]]))

pairs = [(1, 2), (1, 3), (2, 3)]
  cặp kề của nums: (1, 2) -> có
  cặp kề của nums: (2, 3) -> có
True
pairs = [(1, 2), (1, 3)]
  cặp kề của nums: (1, 2) -> có
  cặp kề của nums: (2, 3) -> THIẾU
False
pairs = [(1, 4), (2, 4), (3, 1)]
  cặp kề của nums: (3, 1) -> có
  cặp kề của nums: (1, 2) -> THIẾU
  cặp kề của nums: (2, 4) -> có
False


## 9. Code nộp LeetCode

Ta nộp **Cách 2 (Kahn, hàng đợi luôn đúng một đỉnh)**: đúng ý nghĩa "thứ tự topo duy nhất", dễ giải thích, và không phụ thuộc vào việc đoán cặp kề.

```python
class Solution:
    def sequenceReconstruction(self, nums: List[int], sequences: List[List[int]]) -> bool:
        n = len(nums)
        g = [[] for _ in range(n + 1)]
        indeg = [0] * (n + 1)
        for seq in sequences:
            for a, b in zip(seq, seq[1:]):
                g[a].append(b)
                indeg[b] += 1
        q = deque(v for v in range(1, n + 1) if indeg[v] == 0)
        while len(q) == 1:
            u = q.popleft()
            for v in g[u]:
                indeg[v] -= 1
                if indeg[v] == 0:
                    q.append(v)
        return len(q) == 0
```

(Trên LeetCode `deque` đã được import sẵn; khi chạy ở máy bạn nhớ `from collections import deque`.)

In [18]:
class Solution:
    def sequenceReconstruction(self, nums: List[int], sequences: List[List[int]]) -> bool:
        n = len(nums)
        g = [[] for _ in range(n + 1)]
        indeg = [0] * (n + 1)
        for seq in sequences:
            for a, b in zip(seq, seq[1:]):
                g[a].append(b)
                indeg[b] += 1
        q = deque(v for v in range(1, n + 1) if indeg[v] == 0)
        while len(q) == 1:
            u = q.popleft()
            for v in g[u]:
                indeg[v] -= 1
                if indeg[v] == 0:
                    q.append(v)
        return len(q) == 0


for nums, seqs, expected in examples:
    got = Solution().sequenceReconstruction(nums, seqs)
    print(got)
    assert got == expected

False
False
True


## 10. Vì sao đúng

**Bước 1 — siêu dãy ngắn nhất ↔ thứ tự topo.** Gọi `U` là tập số xuất hiện trong `sequences`. Mọi siêu dãy phải chứa mọi phần tử của `U`; siêu dãy ngắn nhất không có phần tử thừa nên chỉ gồm đúng `U`, mỗi phần tử một lần (có phần tử lặp chỉ làm dài thêm). Siêu dãy là hoán vị của `U` mà mỗi dãy là dãy con của nó ⇔ hoán vị đó thỏa mọi ràng buộc `a` trước `b` của các cặp kề ⇔ là **thứ tự topo** của đồ thị trên `U`. Như vậy "siêu dãy ngắn nhất" = "thứ tự topo của đồ thị".

**Bước 2 — duy nhất ⇔ hàng đợi Kahn luôn một phần tử.** Nếu ở một bước nào đó hàng đợi có hai đỉnh `x`, `y` (đều không có tiền bối chưa xếp), ta có thể chọn `x` rồi hoàn thành, hoặc chọn `y` rồi hoàn thành — cho hai thứ tự topo khác nhau (khác vị trí tương đối của `x`, `y`) ⇒ không duy nhất. Ngược lại nếu luôn chỉ có một lựa chọn thì mọi thứ tự topo đều đồng nhất với dãy lựa chọn đó ⇒ duy nhất.

**Bước 3 — duy nhất thì chính là `nums`.** `nums` thỏa mọi ràng buộc (mỗi dãy là dãy con của `nums`) nên `nums` là một thứ tự topo (trên cả `n` đỉnh). Nếu chỉ có đúng một thứ tự topo thì nó là `nums`.

**Bước 4 — phần tử không xuất hiện.** Nếu có `u` trong `1..n` không thuộc dãy nào thì `indeg[u] = 0` và `u` không có cạnh ra: ở hàng đợi ban đầu `u` luôn đi cùng ít nhất một đỉnh bậc `0` khác (các đỉnh còn lại, nếu có, tạo DAG nên có nguồn; nếu không có đỉnh nào khác thì `n = 1` và khi đó `1` phải xuất hiện trong dãy). Vậy `len(q) >= 2` ngay từ đầu ⇒ `False`, đúng (siêu dãy ngắn nhất không chứa `u`, ngắn hơn `nums`).

**Cách 3.** Đã chứng minh ở mục 7.2: thứ tự topo duy nhất ⇔ mọi cặp kề `nums[i], nums[i+1]` là cạnh.

**Dừng.** Mỗi lần lặp lấy một đỉnh khỏi hàng đợi và mỗi đỉnh vào hàng đợi tối đa một lần (khi `indeg` về `0`), nên có tối đa `n` lần lặp.

## 11. Time / Space Complexity

Gọi `L = sum(len(seq))` (tối đa `10^5`) và `n <= 10^4`.

- **Kahn (Cách 2, 4):** Time `O(n + L)`, Space `O(n + L)`.
- **Cặp kề (Cách 3):** Time `O(n + L)`, Space `O(L)` cho tập cặp.
- **Vét cạn (Cách 1):** Time `O(n! · L)`, chỉ để kiểm chứng với `n` rất nhỏ.

Với ràng buộc của đề, ba cách tuyến tính chạy tức thì (vài mili-giây), vét cạn thì không thể.

## 12. Bộ test

Ta gom các cách vào `impls`. **Oracle** độc lập viết thẳng theo **định nghĩa**: liệt kê mọi hoán vị của các số xuất hiện, giữ hoán vị chứa mọi dãy như dãy con (kiểm tra bằng iterator), đếm xem có đúng một và nó có bằng `nums` hay không. Khác Cách 1 ở cách kiểm tra dãy con (iterator, không dùng bảng `pos`).

In [19]:
def oracle(nums, sequences):
    elems = sorted(set(x for s in sequences for x in s))
    good = [list(p) for p in permutations(elems) if all(is_subsequence(s, p) for s in sequences)]
    return len(good) == 1 and good[0] == nums


impls = {
    "brute": seq_brute,
    "kahn": seq_kahn,
    "adjacent": seq_adjacent,
    "kahn_compare": seq_kahn_compare,
    "leetcode": lambda a, s: Solution().sequenceReconstruction(a, s),
    "submit_kahn": lambda a, s: SolutionKahn().sequenceReconstruction(a, s),
    "submit_adjacent": lambda a, s: SolutionAdjacent().sequenceReconstruction(a, s),
    "submit_compare": lambda a, s: SolutionKahnCompare().sequenceReconstruction(a, s),
}


def check_all(nums, sequences, expected, label):
    assert oracle(nums, sequences) == expected, (label, "oracle")
    for key, fn in impls.items():
        actual = fn(nums, [s[:] for s in sequences])
        assert actual == expected, (label, key, nums, sequences, actual, expected)
    print("PASS", label, "->", expected)


for idx, (nums, seqs, expected) in enumerate(examples, 1):
    check_all(nums, seqs, expected, f"ví dụ {idx}")

PASS ví dụ 1 -> False
PASS ví dụ 2 -> False
PASS ví dụ 3 -> True


### 12.1. Các trường hợp biên

In [20]:
check_all([1], [[1]], True, "n = 1")
check_all([1, 2], [[1, 2]], True, "n = 2, một dãy đầy đủ")
check_all([1, 2], [[1], [2]], False, "n = 2, chỉ hai dãy đơn: thứ tự chưa bị ép")
check_all([1, 2], [[1]], False, "n = 2, thiếu số 2")
check_all([1, 2], [[2]], False, "n = 2, thiếu số 1")
check_all([2, 1], [[2, 1]], True, "n = 2, hoán vị ngược")
check_all([1, 2, 3], [[1, 2, 3]], True, "một dãy đầy đủ = nums")
check_all([1, 2, 3], [[1, 3]], False, "thiếu số 2")
check_all([1, 2, 3], [[1, 2], [2, 3]], True, "chuỗi các cặp kề")
check_all([1, 2, 3], [[1, 2], [1, 3], [2, 3]], True, "ví dụ 3")
check_all([1, 2, 3], [[1, 3], [2, 3]], False, "1 và 2 chưa ép thứ tự")
check_all([3, 1, 2, 4], [[3, 1], [1, 4], [2, 4]], False, "thiếu cạnh 1 -> 2")
check_all([3, 1, 2, 4], [[3, 1, 2], [2, 4]], True, "dãy dài + cặp")
check_all([4, 3, 2, 1], [[4, 3], [3, 2], [2, 1]], True, "ngược hoàn toàn")
check_all([1, 2, 3, 4], [[1, 3], [2, 4], [1, 4]], False, "3 cạnh nhưng không phải các cặp kề của nums")
check_all([1, 2, 3, 4, 5], [[1, 2, 3], [3, 4], [4, 5], [1, 5]], True, "dãy trộn dài và ngắn")
check_all([1, 2, 3], [[1, 2], [1, 2, 3]], True, "cạnh 1->2 lặp ở hai dãy")

PASS n = 1 -> True
PASS n = 2, một dãy đầy đủ -> True
PASS n = 2, chỉ hai dãy đơn: thứ tự chưa bị ép -> False
PASS n = 2, thiếu số 2 -> False
PASS n = 2, thiếu số 1 -> False
PASS n = 2, hoán vị ngược -> True
PASS một dãy đầy đủ = nums -> True
PASS thiếu số 2 -> False
PASS chuỗi các cặp kề -> True
PASS ví dụ 3 -> True
PASS 1 và 2 chưa ép thứ tự -> False
PASS thiếu cạnh 1 -> 2 -> False
PASS dãy dài + cặp -> True
PASS ngược hoàn toàn -> True
PASS 3 cạnh nhưng không phải các cặp kề của nums -> False
PASS dãy trộn dài và ngắn -> True
PASS cạnh 1->2 lặp ở hai dãy -> True


### 12.2. Phản ví dụ cho các cách làm sai

Ba lỗi: (a) **chỉ kiểm tra sắp xếp topo được** (visit hết `n` đỉnh) mà không kiểm tra duy nhất; (b) **bỏ sót số không xuất hiện** (dựng đồ thị chỉ từ các số có mặt); (c) **đếm số cạnh khác nhau `== n - 1`** thay vì kiểm tra đúng các cặp kề của `nums`.

In [21]:
def wrong_visit_all(nums, sequences):
    n = len(nums)
    g = [[] for _ in range(n + 1)]
    indeg = [0] * (n + 1)
    for seq in sequences:
        for a, b in zip(seq, seq[1:]):
            g[a].append(b)
            indeg[b] += 1
    q = deque(v for v in range(1, n + 1) if indeg[v] == 0)
    done = 0
    while q:                              # SAI: cho phép nhiều ứng viên
        u = q.popleft()
        done += 1
        for v in g[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return done == n


def wrong_ignore_missing(nums, sequences):
    nodes = sorted(set(x for s in sequences for x in s))   # SAI: bỏ các số không xuất hiện
    g = {v: [] for v in nodes}
    indeg = {v: 0 for v in nodes}
    for seq in sequences:
        for a, b in zip(seq, seq[1:]):
            g[a].append(b)
            indeg[b] += 1
    q = deque(v for v in nodes if indeg[v] == 0)
    while len(q) == 1:
        u = q.popleft()
        for v in g[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return len(q) == 0


def wrong_count_edges(nums, sequences):
    pairs = set()
    for seq in sequences:
        for a, b in zip(seq, seq[1:]):
            pairs.add((a, b))
    return len(pairs) == len(nums) - 1        # SAI: số cạnh đủ nhưng chưa chắc là các cặp kề của nums


cases = [
    ("không kiểm tra duy nhất", wrong_visit_all, [1, 2, 3], [[1, 2], [1, 3]]),
    ("bỏ sót số vắng mặt", wrong_ignore_missing, [1, 2, 3], [[1, 2]]),
    ("đếm số cạnh", wrong_count_edges, [1, 2, 3, 4], [[1, 3], [2, 4], [1, 4]]),
]
for name, fn, nums, seqs in cases:
    print(f"{name:>24}: sai = {fn(nums, seqs)} | đúng = {Solution().sequenceReconstruction(nums, seqs)}")
    assert fn(nums, seqs) != oracle(nums, seqs)

 không kiểm tra duy nhất: sai = True | đúng = False
      bỏ sót số vắng mặt: sai = True | đúng = False
             đếm số cạnh: sai = True | đúng = False


### 12.3. Đối chiếu ngẫu nhiên (có seed) với oracle

Sinh đầu vào **đúng ràng buộc**: `nums` là hoán vị ngẫu nhiên của `1..n` (`n <= 7`); các dãy là **dãy con** của `nums`, đôi một khác nhau. Có hai kiểu để có cả đáp án `True` lẫn `False`:

- kiểu "dày": mỗi cặp kề của `nums` được thêm dưới dạng dãy `[a, b]` với xác suất cao, cộng thêm vài dãy con ngẫu nhiên;
- kiểu "thưa": chỉ vài dãy con ngẫu nhiên (hầu hết `False`).

So mọi cách với oracle và thống kê số `True`/`False`.

In [22]:
def random_subsequence(rng, nums):
    k = rng.randint(1, len(nums))
    idx = sorted(rng.sample(range(len(nums)), k))
    return [nums[i] for i in idx]


def gen(rng):
    n = rng.randint(1, 7)
    nums = list(range(1, n + 1))
    rng.shuffle(nums)
    seqs = []
    if rng.random() < 0.6:
        p = rng.choice([0.8, 0.9, 1.0])
        for i in range(n - 1):
            if rng.random() < p:
                seqs.append([nums[i], nums[i + 1]])
        if n == 1 or not seqs:
            seqs.append(random_subsequence(rng, nums))
        for _ in range(rng.randint(0, 2)):
            seqs.append(random_subsequence(rng, nums))
    else:
        for _ in range(rng.randint(1, 4)):
            seqs.append(random_subsequence(rng, nums))
    uniq = []
    seen = set()
    for s in seqs:
        if tuple(s) not in seen:
            seen.add(tuple(s))
            uniq.append(s)
    rng.shuffle(uniq)
    return nums, uniq


rng = random.Random(444)
stats = {True: 0, False: 0}
for trial in range(2500):
    nums, seqs = gen(rng)
    assert sorted(nums) == list(range(1, len(nums) + 1))
    assert all(is_subsequence(s, nums) for s in seqs) and len(set(map(tuple, seqs))) == len(seqs)
    expected = oracle(nums, seqs)
    stats[expected] += 1
    for key, fn in impls.items():
        assert fn(nums, [s[:] for s in seqs]) == expected, (key, nums, seqs)
print("2500 test OK; True:", stats[True], "| False:", stats[False])
assert stats[True] > 300 and stats[False] > 300

2500 test OK; True: 1816 | False: 684


### 12.4. Hiệu năng với `n = 10^4`, `L = 10^5`

Bỏ vét cạn. Ta dựng ba loại input lớn: (a) mọi cặp kề của `nums` (đáp án `True`); (b) bỏ **một** cặp kề ở giữa (đáp án `False`); (c) một dãy duy nhất bằng `nums` (đáp án `True`). Tổng độ dài các dãy không vượt `10^5`.

In [23]:
rng = random.Random(4440)
n = 10000
nums = list(range(1, n + 1))
rng.shuffle(nums)
all_pairs = [[nums[i], nums[i + 1]] for i in range(n - 1)]
rng.shuffle(all_pairs)
drop = rng.randint(1, n - 3)
missing_one = [p for p in all_pairs if not (p[0] == nums[drop] and p[1] == nums[drop + 1])]
big_cases = [
    ("mọi cặp kề", nums, all_pairs, True),
    ("thiếu một cặp", nums, missing_one, False),
    ("một dãy = nums", nums, [nums[:]], True),
]
fast = {k: v for k, v in impls.items() if k != "brute"}
for label, nm, sq, expected in big_cases:
    assert sum(len(s) for s in sq) <= 10**5
    for key, fn in fast.items():
        t0 = time.perf_counter()
        got = fn(nm, [s[:] for s in sq])
        dt = time.perf_counter() - t0
        assert got == expected, (label, key)
    print(f"{label}: tất cả các cách đều trả {expected}, cách chậm nhất cỡ {dt * 1000:.0f} ms")

mọi cặp kề: tất cả các cách đều trả True, cách chậm nhất cỡ 6 ms
thiếu một cặp: tất cả các cách đều trả False, cách chậm nhất cỡ 5 ms
một dãy = nums: tất cả các cách đều trả True, cách chậm nhất cỡ 4 ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ kiểm tra "sắp xếp topo được" mà quên tính duy nhất

**Code sai:** `while q:` rồi đếm số đỉnh đã lấy ra, trả `done == n`.

**Vì sao sai:** mọi input hợp lệ của đề **luôn** sắp xếp topo được (vì là dãy con của `nums`), nên điều kiện này luôn đúng ⇒ luôn trả `True`, kể cả ví dụ 1 (`[[1,2],[1,3]]`) có hai siêu dãy ngắn nhất.

**Code đúng:** `while len(q) == 1:` và trả `len(q) == 0` (hoặc báo `False` ngay khi `len(q) > 1`).

In [24]:
nums, seqs = [1, 2, 3], [[1, 2], [1, 3]]
print("sai :", wrong_visit_all(nums, seqs))
print("đúng:", Solution().sequenceReconstruction(nums, seqs))

sai : True
đúng: False


### Lỗi 2: Bỏ sót những số không xuất hiện trong dãy nào

**Code sai:** chỉ dựng đồ thị trên những số xuất hiện trong `sequences`.

**Vì sao sai:** ví dụ 2 (`nums = [1,2,3]`, `[[1,2]]`): các số có mặt là `{1, 2}`, thứ tự `1 → 2` duy nhất nên code sai trả `True`. Nhưng siêu dãy ngắn nhất là `[1, 2]` (dài `2`) chứ không phải `nums` (dài `3`) ⇒ phải `False`.

**Code đúng:** xét **mọi** `1..n` (mảng `indeg` dài `n + 1`); số vắng mặt có bậc vào `0` và làm hàng đợi có thêm một ứng viên.

In [25]:
nums, seqs = [1, 2, 3], [[1, 2]]
print("sai :", wrong_ignore_missing(nums, seqs))
print("đúng:", Solution().sequenceReconstruction(nums, seqs))

sai : True
đúng: False


### Lỗi 3: Đếm số cạnh thay vì kiểm tra đúng các cặp kề của `nums`

**Code sai:** `return len(pairs) == n - 1`.

**Vì sao sai:** có thể có `n - 1` cạnh khác nhau nhưng **không** phải các cặp kề của `nums`. Ví dụ `nums = [1,2,3,4]`, `[[1,3],[2,4],[1,4]]`: có `3` cạnh `1→3, 2→4, 1→4` = `n - 1`, nhưng chưa có cạnh `1→2`, `2→3`, `3→4` nên thứ tự chưa bị ép (`2` có thể ở trước hoặc sau `3`, ...).

**Code đúng:** `all((nums[i], nums[i+1]) in pairs for i in range(n - 1))`.

In [26]:
nums, seqs = [1, 2, 3, 4], [[1, 3], [2, 4], [1, 4]]
print("sai :", wrong_count_edges(nums, seqs))
print("đúng:", Solution().sequenceReconstruction(nums, seqs), "| oracle:", oracle(nums, seqs))

sai : True
đúng: False | oracle: False


### Lỗi 4: Nhầm chỉ số khi giá trị bắt đầu từ `1`

**Code sai:** cấp `indeg = [0] * n` rồi dùng `indeg[b] += 1` với `b = n`.

**Vì sao sai:** giá trị lớn nhất là `n` nhưng mảng chỉ có chỉ số `0..n-1` ⇒ `IndexError`. Hoặc trừ `1` ở chỗ này mà quên ở chỗ khác ⇒ lệch cạnh một cách im lặng. Cách an toàn: mảng dài `n + 1` (bỏ ô `0`), hoặc luôn dùng `a - 1`, `b - 1` ở mọi nơi.

In [27]:
def wrong_index(nums, sequences):
    n = len(nums)
    indeg = [0] * n                      # SAI: thiếu một ô
    for seq in sequences:
        for a, b in zip(seq, seq[1:]):
            indeg[b] += 1
    return indeg


try:
    wrong_index([1, 2, 3], [[1, 2], [2, 3]])
except IndexError as e:
    print("IndexError:", e)
print("đúng: dùng mảng dài n + 1 ->", [0] * (3 + 1))

IndexError: list index out of range
đúng: dùng mảng dài n + 1 -> [0, 0, 0, 0]


## 14. Mẹo & nhận dạng pattern

**Dấu hiệu nhận dạng:** "thứ tự", "đứng trước", "có duy nhất một thứ tự không", "dãy con ép thứ tự" ⇒ **sắp xếp topo**. Cứ thấy "unique" thì nhớ: duy nhất ⇔ **tại mọi bước chỉ có đúng một lựa chọn** (hàng đợi Kahn có độ dài `1`).

Bảng nhắc:

| Câu hỏi | Gợi ý |
|---|---|
| Có thể xếp thứ tự được không (có chu trình không)? | Kahn: đếm số đỉnh lấy ra `== n` |
| Thứ tự có duy nhất không? | Hàng đợi luôn đúng `1` phần tử |
| Muốn thứ tự nhỏ nhất theo từ điển? | Dùng `heapq` thay `deque` |
| Dựng cạnh từ dãy | Chỉ cần các cặp kề `zip(seq, seq[1:])` |
| Giá trị `1..n` | Mảng dài `n + 1` hoặc trừ `1` nhất quán |

Bài cùng họ: **207 Course Schedule**, **210 Course Schedule II**, **269 Alien Dictionary**, **310 Minimum Height Trees**, **1203 Sort Items by Groups Respecting Dependencies**.

## 15. Biến thể

### 15.1. Trả về **một** siêu dãy ngắn nhất bất kỳ (nếu có)

Vẫn dùng Kahn nhưng cho phép nhiều ứng viên: lấy lần lượt đầu hàng đợi, ghi vào kết quả. Kết quả là một thứ tự topo của các số xuất hiện (LeetCode 210/269 cùng dạng).

In [28]:
def one_shortest_supersequence(sequences):
    nodes = sorted(set(x for s in sequences for x in s))
    g = {v: [] for v in nodes}
    indeg = {v: 0 for v in nodes}
    for seq in sequences:
        for a, b in zip(seq, seq[1:]):
            g[a].append(b)
            indeg[b] += 1
    q = deque(v for v in nodes if indeg[v] == 0)
    order = []
    while q:
        u = q.popleft()
        order.append(u)
        for v in g[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return order


res = one_shortest_supersequence([[1, 2], [1, 3]])
print(res)
assert all(is_subsequence(s, res) for s in [[1, 2], [1, 3]]) and len(res) == 3
assert one_shortest_supersequence([[1, 2], [1, 3], [2, 3]]) == [1, 2, 3]

[1, 2, 3]


### 15.2. Đếm **số** siêu dãy ngắn nhất (n nhỏ)

Dùng quy hoạch động theo bitmask: `dp[mask]` = số cách xếp các đỉnh trong `mask` thành tiền tố hợp lệ. Thêm đỉnh `v` được nếu mọi tiền bối của `v` đã nằm trong `mask`. Dùng để hiểu "không duy nhất" nhiều đến mức nào, và để kiểm chứng chéo với oracle.

In [29]:
def count_shortest(sequences):
    nodes = sorted(set(x for s in sequences for x in s))
    idx = {v: i for i, v in enumerate(nodes)}
    m = len(nodes)
    pre = [0] * m                                # pre[i] = mask các tiền bối của nodes[i]
    for seq in sequences:
        for a, b in zip(seq, seq[1:]):
            pre[idx[b]] |= 1 << idx[a]
    dp = [0] * (1 << m)
    dp[0] = 1
    for mask in range(1 << m):
        if dp[mask] == 0:
            continue
        for v in range(m):
            if not (mask >> v) & 1 and (pre[v] & ~mask) == 0:
                dp[mask | (1 << v)] += dp[mask]
    return dp[(1 << m) - 1]


assert count_shortest([[1, 2], [1, 3]]) == 2
assert count_shortest([[1, 2], [1, 3], [2, 3]]) == 1
assert count_shortest([[1], [2], [3]]) == 6
rng = random.Random(4441)
for _ in range(300):
    nums, seqs = gen(rng)
    assert (count_shortest(seqs) == 1 and one_shortest_supersequence(seqs) == nums) == oracle(nums, seqs)
print("đếm DP khớp với oracle:", count_shortest([[1], [2], [3]]), "thứ tự cho 3 số tự do")

đếm DP khớp với oracle: 6 thứ tự cho 3 số tự do


### 15.3. Thứ tự nhỏ nhất theo từ điển

Thay `deque` bằng `heapq` (hàng đợi ưu tiên): mỗi lần lấy đỉnh **nhỏ nhất** trong số các đỉnh bậc vào `0` ⇒ thứ tự topo nhỏ nhất theo từ điển (như LeetCode 1203/ "Alien Dictionary" cần thứ tự cụ thể).

In [30]:
import heapq


def lexicographically_smallest(sequences):
    nodes = sorted(set(x for s in sequences for x in s))
    g = {v: [] for v in nodes}
    indeg = {v: 0 for v in nodes}
    for seq in sequences:
        for a, b in zip(seq, seq[1:]):
            g[a].append(b)
            indeg[b] += 1
    heap = [v for v in nodes if indeg[v] == 0]
    heapq.heapify(heap)
    order = []
    while heap:
        u = heapq.heappop(heap)
        order.append(u)
        for v in g[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                heapq.heappush(heap, v)
    return order


assert lexicographically_smallest([[3, 1], [3, 2]]) == [3, 1, 2]
assert lexicographically_smallest([[1, 2], [1, 3]]) == [1, 2, 3]
print(lexicographically_smallest([[3, 1], [3, 2]]))

[3, 1, 2]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn hoán vị | Liệt kê hoán vị hợp lệ, đếm | `O(n! · L)` | `O(n)` | Làm oracle với `n` rất nhỏ |
| 2. Kahn, hàng đợi 1 phần tử | Thứ tự topo duy nhất ⇔ luôn một ứng viên | `O(n + L)` | `O(n + L)` | **Lời giải nộp**, dễ chứng minh |
| 3. Cặp kề của `nums` | Mọi `nums[i]→nums[i+1]` phải là cạnh trực tiếp | `O(n + L)` | `O(L)` | Ngắn nhất; cần giả thiết dãy con của `nums` |
| 4. Kahn so với `nums` | Mỗi bước hàng đợi = `[nums[i]]` | `O(n + L)` | `O(n + L)` | Khi muốn so trực tiếp, không dựa vào giả thiết |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `list(zip([5, 3, 8], [5, 3, 8][1:]))`, `is_subsequence([2, 4], [1, 2, 3, 4])`, `is_subsequence([4, 2], [1, 2, 3, 4])`, `Solution().sequenceReconstruction([1, 2], [[1, 2]])`, `Solution().sequenceReconstruction([2, 1, 3], [[2, 1], [2, 3]])`.

**Bài 2 (tính tay).** Với `nums = [2, 1, 3]`, `sequences = [[2, 1], [1, 3]]`: lập `g`, `indeg` (chỉ cho `1..3`) và bảng Kahn từng bước (hàng đợi trước, đỉnh lấy ra, hàng đợi sau) và nêu kết quả.

**Bài 3 (điền TODO).** Hoàn thành `unique_todo(nums, sequences)` theo Cách 2.

In [31]:
def unique_todo(nums, sequences):
    n = len(nums)
    g = [[] for _ in range(n + 1)]
    indeg = [0] * (n + 1)
    # TODO 1: với mỗi seq, với mỗi cặp kề (a, b): g[a].append(b); indeg[b] += 1
    # TODO 2: q = deque các v trong 1..n có indeg[v] == 0
    # TODO 3: while len(q) == 1: lấy u ra, giảm indeg của các v kề, v về 0 thì đưa vào q
    # TODO 4: return len(q) == 0
    return None

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `unique_adjacent2(nums, sequences)` theo Cách 3 nhưng dùng **mảng vị trí** `pos[v]` thay cho `set` cặp: với mỗi cặp kề `(a, b)` của mọi dãy, nếu `pos[b] == pos[a] + 1` thì đánh dấu `ok[pos[a]] = True`; đáp án là `all(ok)` (cho `n >= 2`), nhớ xử lý `n == 1` và số vắng mặt.

**Bài 5 (biến thể / mini challenge).** (a) Viết `count_orders(sequences)` (đếm số thứ tự topo) cho `m <= 10` đỉnh bằng DP bitmask, rồi kiểm tra `count_orders([[1],[2],[3],[4]]) == 24`. (b) Viết `is_unique_order(n, sequences)` cho trường hợp **tổng quát** — các dãy **không** được đảm bảo là dãy con của một hoán vị chung (có thể có chu trình): trả về `False` nếu có chu trình hoặc có nhiều hơn một thứ tự, ngược lại `True`. Gợi ý: đếm số đỉnh đã lấy ra.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[(5, 3), (3, 8)]`, `True`, `False`, `True`, `False` (sau khi lấy `2`, cả `1` lẫn `3` cùng có bậc vào `0`). Bài 3: cặp kề `zip(seq, seq[1:])`.
- **Mức 2 (vừa):** Bài 2: cạnh `2→1`, `1→3`; `indeg = [_, 1, 0, 1]`; `q = [2]` → lấy `2`, `indeg[1] → 0` ⇒ `q = [1]` → lấy `1`, `indeg[3] → 0` ⇒ `q = [3]` → lấy `3`, `q = []` ⇒ `True`. Bài 4: `ok = [False] * (n - 1)`.
- **Mức 3 (sâu):** Bài 5(a): `dp[mask | 1<<v] += dp[mask]` khi `pre[v] & ~mask == 0`. 5(b): thêm điều kiện cuối `done == n` (số đỉnh lấy ra đủ `n`, nếu không có chu trình hoặc đỉnh bị kẹt) và trả `False` ngay khi `len(q) != 1` ở một bước còn đỉnh chưa xếp; với `n` đỉnh tự do (không dãy nào chứa) hàng đợi có `n` đỉnh ⇒ `False` (nếu `n >= 2`).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [32]:
# Bài 1
assert list(zip([5, 3, 8], [5, 3, 8][1:])) == [(5, 3), (3, 8)]
assert is_subsequence([2, 4], [1, 2, 3, 4]) is True
assert is_subsequence([4, 2], [1, 2, 3, 4]) is False
assert Solution().sequenceReconstruction([1, 2], [[1, 2]]) is True
assert Solution().sequenceReconstruction([2, 1, 3], [[2, 1], [2, 3]]) is False   # thiếu cạnh 1->3
assert Solution().sequenceReconstruction([2, 1, 3], [[2, 1], [1, 3]]) is True    # bài 2


# Bài 3
def unique_todo_ans(nums, sequences):
    n = len(nums)
    g = [[] for _ in range(n + 1)]
    indeg = [0] * (n + 1)
    for seq in sequences:
        for a, b in zip(seq, seq[1:]):
            g[a].append(b)
            indeg[b] += 1
    q = deque(v for v in range(1, n + 1) if indeg[v] == 0)
    while len(q) == 1:
        u = q.popleft()
        for v in g[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return len(q) == 0


for nums, seqs, expected in examples:
    assert unique_todo_ans(nums, seqs) == expected


# Bài 4
def unique_adjacent2(nums, sequences):
    n = len(nums)
    pos = [0] * (n + 1)
    for i, v in enumerate(nums):
        pos[v] = i
    if set(x for s in sequences for x in s) != set(nums):
        return False
    ok = [False] * max(n - 1, 0)
    for seq in sequences:
        for a, b in zip(seq, seq[1:]):
            if pos[b] == pos[a] + 1:
                ok[pos[a]] = True
    return all(ok)


rng = random.Random(4442)
for _ in range(500):
    nums, seqs = gen(rng)
    assert unique_adjacent2(nums, seqs) == oracle(nums, seqs)
for nums, seqs, expected in examples:
    assert unique_adjacent2(nums, seqs) == expected


# Bài 5
def count_orders(sequences):
    return count_shortest(sequences)


assert count_orders([[1], [2], [3], [4]]) == 24
assert count_orders([[1, 2, 3, 4]]) == 1


def is_unique_order(n, sequences):
    g = [[] for _ in range(n + 1)]
    indeg = [0] * (n + 1)
    for seq in sequences:
        for a, b in zip(seq, seq[1:]):
            g[a].append(b)
            indeg[b] += 1
    q = deque(v for v in range(1, n + 1) if indeg[v] == 0)
    done = 0
    while q:
        if len(q) > 1:
            return False
        u = q.popleft()
        done += 1
        for v in g[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return done == n


assert is_unique_order(3, [[1, 2], [2, 3]]) is True
assert is_unique_order(3, [[1, 2], [1, 3]]) is False
assert is_unique_order(2, [[1, 2], [2, 1]]) is False      # có chu trình
assert is_unique_order(3, [[1, 2]]) is False              # số 3 tự do
print("Tất cả đáp án bài tập đều đúng.")

Tất cả đáp án bài tập đều đúng.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (thứ tự topo **duy nhất** bằng Kahn)

```python
class Solution:
    def sequenceReconstruction(self, nums: List[int], sequences: List[List[int]]) -> bool:
        n = len(nums)
        g = [[] for _ in range(n + 1)]                 # giá trị 1..n -> mảng dài n + 1
        indeg = [0] * (n + 1)
        for seq in sequences:
            for a, b in zip(seq, seq[1:]):             # chỉ cần các cặp kề
                g[a].append(b)
                indeg[b] += 1
        q = deque(v for v in range(1, n + 1) if indeg[v] == 0)
        while len(q) == 1:                             # còn đúng một ứng viên
            u = q.popleft()
            for v in g[u]:
                indeg[v] -= 1
                if indeg[v] == 0:
                    q.append(v)
        return len(q) == 0                             # 0: xếp hết duy nhất; >= 2: có hai lựa chọn
```

**Checklist**

- [ ] Xét **mọi** số `1..n` (kể cả số không xuất hiện trong dãy nào)?
- [ ] Điều kiện vòng lặp là `len(q) == 1` (không phải `while q`)? Trả `len(q) == 0`?
- [ ] Dựng cạnh chỉ từ **cặp kề** `zip(seq, seq[1:])`; mảng `indeg`, `g` dài `n + 1`?
- [ ] Biết vì sao thứ tự duy nhất đó chính là `nums` (nums luôn hợp lệ)?
- [ ] Đã thử `n = 1`, hai dãy đơn `[[1],[2]]`, thiếu một số, thiếu đúng một cặp kề, ví dụ 1-2-3?
- [ ] Skeleton đúng `def sequenceReconstruction(self, nums: List[int], sequences: List[List[int]]) -> bool`? (Bài Premium: đối chiếu lại đề gốc nếu có tài khoản.)